# Section 1 — Single Agents Fundamentals

**Microsoft Agent Framework + Azure AI Foundry Workshop**

Welcome to the first hands-on section of the workshop! In this notebook you will build your first AI agent with the Microsoft Agent Framework, give it tools, hold a multi-turn conversation with it, and add memory.

## 🎯 Learning Objectives

By the end of this notebook, you will be able to:

1. **Create** an `Agent` backed by a Foundry-hosted model via `FoundryChatClient`.
2. **Equip** the agent with Python function tools using simple type-hinted functions.
3. **Hold** a multi-turn conversation that preserves context across turns.
4. **Persist** memory across sessions with a `ContextProvider`.
5. **Stream** responses for a more interactive UX.

## 🧩 What we are NOT covering here

- The **hosted Foundry Agent Service** (`FoundryAgentClient` / `AzureAIAgentClient`) — out of scope for this workshop. We use the **client-side `Agent` class with a chat client** (`FoundryChatClient`), which runs the agent loop locally and only calls Foundry for model inference.
- DevUI — covered in Section 2.
- Workflows and orchestrations — Sections 3 & 4.

## 📦 Prerequisites

- Python 3.10+
- `agent-framework` GA v1.0 installed (`pip install agent-framework`)
- Access to the **shared Foundry project endpoint** (provided by the instructor)
- Logged in via `az login` so `AzureCliCredential` works
- A `.env` file in this folder with at least:

  ```env
  FOUNDRY_PROJECT_ENDPOINT=https://<shared-project>.services.ai.azure.com/api/projects/<project>
  FOUNDRY_MODEL=gpt-4o
  ```

> 💡 If something fails in the very first cell, raise your hand — we'll resolve setup issues together before continuing.

## 0. Setup — verify your environment

This cell loads `.env`, prints which Foundry endpoint and model you will use, and confirms the SDK is importable. **Run it first.**

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT", "<not set>")
model = os.getenv("FOUNDRY_MODEL", "gpt-4o")
print(f"Foundry endpoint: {endpoint}")
print(f"Model:            {model}")

import agent_framework  # noqa: F401
from agent_framework import Agent
from agent_framework.foundry import FoundryChatClient
from azure.identity.aio import AzureCliCredential

print("✅ agent-framework imports OK")

## 1. Hello, Agent — your first run

An **`Agent`** in Microsoft Agent Framework is a thin wrapper around three things:

| Concept | What it is |
|---------|------------|
| **Client** | Where the model lives (here: `FoundryChatClient` calling a Foundry-deployed model). |
| **Instructions** | The system prompt — the agent's persona / behavior. |
| **Tools** *(optional)* | Python functions the model can call. We'll add these in Task 2. |

We use `AzureCliCredential` for auth — meaning the SDK uses the identity from your last `az login`.

### ▶️ Task 1
Run the cell below and read the output. Then change the **instructions** to make the agent answer like a pirate, and re-run. Notice how the personality changes with no other code change.

In [ ]:
async def hello_agent():
    async with AzureCliCredential() as credential, Agent(
        client=FoundryChatClient(model=model, credential=credential),
        name="HelloAgent",
        instructions="You are a friendly assistant. Keep replies short and helpful.",
    ) as agent:
        result = await agent.run("In one sentence, what is the Microsoft Agent Framework?")
        print(result.text)

await hello_agent()

### 💡 Why `async with`?

`AzureCliCredential` and `Agent` both manage resources (HTTP clients, token caches). The `async with` pattern guarantees they are cleanly disposed when the block exits — important when you scale to many concurrent agents later.

### ✏️ Mini-exercise

Replace the instructions with: `"You are a senior Python developer. Always answer with a code example first, then a one-line explanation."` and ask: *"How do I read a JSON file?"*

Notice that **the same model + same question** produces a very different answer when the instructions change. This is the cheapest, fastest way to shape agent behavior.

## 2. Adding Tools — let the agent take actions

An agent without tools can only talk. With tools, it can **act**: query an API, hit a database, run a calculation, send an email.

In Agent Framework, a tool is just a Python function. The framework introspects the **type hints** and **docstring** to build the schema sent to the model. The model decides when to call it, the framework executes it, and the result is fed back into the conversation — a loop the framework runs for you.

Here we give the agent two mock tools: `get_weather` and `get_forecast`.

In [ ]:
def get_weather(city: str) -> str:
    """Return the current weather for a given city."""
    # In a real workshop you'd call an API; here we mock for determinism.
    fake = {"Seattle": "55°F, drizzle", "Paris": "68°F, sunny", "Tokyo": "72°F, clear"}
    return fake.get(city, f"73°F and sunny in {city}")

def get_forecast(city: str, days: int = 3) -> str:
    """Return a short forecast for a given city for the next `days` days."""
    return f"{city} forecast for the next {days} days: mostly sunny, mild temperatures."

async def weather_agent():
    async with AzureCliCredential() as credential, Agent(
        client=FoundryChatClient(model=model, credential=credential),
        name="WeatherAgent",
        instructions=(
            "You are a weather assistant. Use the provided tools to answer questions "
            "about current weather and forecasts. Always cite the city."
        ),
        tools=[get_weather, get_forecast],
    ) as agent:
        result = await agent.run("What's the weather in Seattle, and what should I expect for the next 5 days?")
        print(result.text)

await weather_agent()

### 🔍 What just happened

Behind the scenes the framework ran a small loop:

1. Send the user message + tool schemas to the model.
2. Model responds with a **tool call** (e.g., `get_weather(city="Seattle")`).
3. Framework executes the Python function, captures the return value.
4. Framework sends the tool result back to the model.
5. Model produces the final natural-language answer.

You don't write any of that loop — that's the value of the framework. You'll *see* this loop visually in Section 2 with DevUI.

### ✏️ Mini-exercise

Add a new tool `get_air_quality(city: str) -> str` that returns a string like `"AQI 42 (Good)"`. Re-run with the question: *"What's the weather and air quality in Tokyo?"* — does the model call **both** tools?

## 3. Multi-turn conversations

Each call to `agent.run()` is **stateless by default** — it does not remember previous turns. To carry context forward, use a **thread**: a lightweight object that accumulates messages.

Threads are the building block for chatbot-style experiences.

In [ ]:
async def multi_turn():
    async with AzureCliCredential() as credential, Agent(
        client=FoundryChatClient(model=model, credential=credential),
        name="TravelBuddy",
        instructions="You are a concise travel assistant. Remember user preferences across the conversation.",
    ) as agent:
        thread = agent.get_new_thread()

        r1 = await agent.run("I'm planning a trip to Japan in spring. I love food and quiet places.", thread=thread)
        print("Turn 1:", r1.text, "\n")

        r2 = await agent.run("Suggest one city that fits.", thread=thread)
        print("Turn 2:", r2.text, "\n")

        r3 = await agent.run("What's one specific dish I should try there?", thread=thread)
        print("Turn 3:", r3.text)

await multi_turn()

Notice in Turn 2 the agent already "knows" you like food and quiet places — even though you didn't repeat it — because the `thread` carries the full message history.

### ✏️ Mini-exercise

Run the same three turns but **without** passing `thread=thread`. Each turn should feel like talking to a stranger — proving the thread is what gives continuity.

## 4. Streaming responses

For a chat UX, you usually want tokens to appear as they're generated, not all at once. Use `agent.run_stream(...)` and iterate over the chunks.

In [ ]:
async def stream_demo():
    async with AzureCliCredential() as credential, Agent(
        client=FoundryChatClient(model=model, credential=credential),
        name="Storyteller",
        instructions="You are a creative storyteller. Keep stories under 5 sentences.",
    ) as agent:
        async for update in agent.run_stream("Tell me a tiny story about a curious robot."):
            if update.text:
                print(update.text, end="", flush=True)
        print()

await stream_demo()

## 5. Persistent memory with a `ContextProvider`

A **thread** keeps state for the duration of one conversation. A **`ContextProvider`** is the framework's hook for *durable* memory — facts, preferences, summaries — that should persist across conversations and be injected into the prompt automatically.

Below we use the simplest possible provider: an in-memory store of user preferences. In production you'd back it with Cosmos DB, Redis, or Azure AI Search.

In [ ]:
from agent_framework import ChatMessage, Context, ContextProvider, Role

class PreferencesMemory(ContextProvider):
    """Tiny in-memory provider that injects known user preferences as a system message."""

    def __init__(self) -> None:
        self._prefs: dict[str, str] = {}

    def remember(self, key: str, value: str) -> None:
        self._prefs[key] = value

    async def invoking(self, messages, **kwargs) -> Context:  # called by the framework before each model call
        if not self._prefs:
            return Context()
        bullets = "\n".join(f"- {k}: {v}" for k, v in self._prefs.items())
        return Context(messages=[ChatMessage(role=Role.SYSTEM, text=f"Known user preferences:\n{bullets}")])

async def memory_demo():
    memory = PreferencesMemory()
    memory.remember("diet", "vegetarian")
    memory.remember("favorite_cuisine", "Thai")

    async with AzureCliCredential() as credential, Agent(
        client=FoundryChatClient(model=model, credential=credential),
        name="FoodieAgent",
        instructions="You are a friendly food recommender. Always honour the user's known preferences.",
        context_providers=[memory],
    ) as agent:
        r = await agent.run("Suggest a dinner for tonight.")
        print(r.text)

await memory_demo()

Even though we never said *"I'm vegetarian and like Thai food"* in the user message, the suggestion respects both — because `PreferencesMemory.invoking` injected them as a system message before the model call.

### ✏️ Mini-exercise

Add a third preference (`memory.remember("allergies", "peanuts")`) and re-ask. The recommendation should avoid peanuts.

> 🧠 **Connection to later sections:** This is the same hook used by **Azure AI Search context providers** (Section 6 / RAG-style augmentation) — durable knowledge injected on every call without bloating the user-facing prompt.

## ✅ Recap & what's next

You now know how to:

- Create an `Agent` with a Foundry-backed model.
- Add Python function tools.
- Hold multi-turn conversations using threads.
- Stream responses.
- Inject durable memory via a `ContextProvider`.

**Next: Section 2 — Foundry-Backed Agents + DevUI.** We will take this same agent and explore it in DevUI to *see* the tool-call loop, message history, and timing — the same agent we just wrote, with no code changes.

### 🏆 Stretch challenge (if you finish early)

Combine everything: build an agent named `TripPlanner` with

- 2 tools: `get_weather(city)` and `get_currency(country) -> str`
- a `ContextProvider` that remembers `home_currency` and `dietary_restrictions`
- a thread carrying a 3-turn planning conversation: *(1)* destination idea, *(2)* what to pack, *(3)* a budget breakfast suggestion in the local currency.